# Touch RF Training — Standalone Notebook

Train + evaluate **touch modality riêng biệt** (Stage 3 isolated) khỏi backbone inertial. Mục đích:

- **Iteration nhanh**: touch RF train trên CPU mất vài giây/owner, một full run 10 seeds × 23 users hết ~1–3 phút (so với 30–40 phút cho pipeline đầy đủ phải train cả backbone CNN).
- **Hyperparameter ablation**: dễ dàng grid-search `min_samples_leaf`, `n_estimators`, `max_features`, ... mà không phải re-train backbone.
- **Metrics so sánh được**: session-split dùng cùng logic + seed với pipeline chính → kết quả `auc_test` baseline (msl=1) phải khớp với cột `auc_touch` của `summary.csv` từ pipeline chính.

## Yêu cầu input

Folder `processed_data/` với cấu trúc giống pipeline chính:

```
processed_data/
├── user1/
│   ├── y_inertial.npy             ← chỉ cần để lấy session list
│   ├── y_walking.npy              ← chỉ cần để lấy session list
│   ├── touch_session_features.csv ← BẮT BUỘC, 48 cột + session_id
│   ├── tap_gestures.csv
│   └── scroll_gestures.csv
├── user2/
└── ...
```

Notebook này **không cần** `X_*.npy` (inertial windows) — chỉ đọc session ID strings từ `y_*.npy` để khớp split với pipeline chính.

## Output

Sẽ tạo ra các folder `results_touch_<config>/` chứa:
- `summary.csv` — per (run, owner) AUC/EER/FAR/FRR
- `final_report.txt` — aggregate + per-user ranking
- `config.json` — hyperparams đã dùng

## Cách dùng

1. Mở trong Colab (không cần GPU — touch là CPU)
2. Chỉnh `DATA_DIR` ở §3
3. Runtime → Run all
4. §7 chạy baseline, §8 grid-search MSL, §9 so sánh kết quả


## §1. Workspace

Touch RF train trên CPU — **không cần GPU**.


In [ ]:
import os
from pathlib import Path

WORKSPACE = Path("/content/touch_pipeline")
WORKSPACE.mkdir(exist_ok=True)
os.chdir(WORKSPACE)
print(f"Workspace: {WORKSPACE}")


## §2. Mount Google Drive

Bỏ qua nếu data đã upload trực tiếp lên Colab.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Đã mount /content/drive")
except ImportError:
    print("Không phải Colab — bỏ qua mount.")


## §3. Cấu hình

Chỉnh `DATA_DIR` cho đúng vị trí `processed_data/` của bạn.

`MSL_GRID` là list các giá trị `min_samples_leaf` để ablation ở §8 — mặc định thử [1, 2, 3, 5].


In [ ]:
# Đường dẫn data
DATA_DIR = "/content/drive/MyDrive/DATN/processed"
# Hoặc nếu upload trực tiếp lên Colab:
# DATA_DIR = "/content/processed_data"

# Số seed để chạy (median ổn định với N_RUNS ≥ 10)
N_RUNS = 10

# Mode: lấy session list từ file nào
#   'walking' : đọc y_walking.npy
#   'all'     : đọc y_inertial.npy (default — khớp pipeline chính)
CONTEXT_MODE = "all"

# Grid để ablation min_samples_leaf ở §8
MSL_GRID = [1, 2, 3, 5]

# Hyperparams cố định trong ablation MSL (chỉnh nếu muốn ablation chiều khác)
BASE_HP = {
    "n_estimators":  200,
    "max_features":  "sqrt",
    "class_weight":  "balanced",
    "pool_size":     100,
}

# Verify
p = Path(DATA_DIR)
assert p.exists(), f"Không tìm thấy {DATA_DIR}"
users_found = sorted([d.name for d in p.iterdir() if d.is_dir()])
print(f"DATA_DIR      = {DATA_DIR}")
print(f"N_RUNS        = {N_RUNS}")
print(f"CONTEXT_MODE  = {CONTEXT_MODE}")
print(f"MSL_GRID      = {MSL_GRID}")
print(f"BASE_HP       = {BASE_HP}")
print(f"Users ({len(users_found)}): {users_found}")
assert len(users_found) >= 2, "Cần >= 2 user"


## §4. Ghi source code ra disk

Tách thành 3 module ghi từ notebook ra file `.py`:

| File | Nhiệm vụ |
|---|---|
| `touch_features.py` | Schema 48-D + đọc `touch_session_features.csv` |
| `metrics.py`        | AUC, EER, FAR, FRR (robust với edge case) |
| `touch_train.py`    | Orchestrator: split → train per-owner → eval |

Có thể chỉnh trực tiếp các file này (ngoài notebook) khi cần debug nhanh.


In [ ]:
%%writefile touch_features.py
"""
touch_features.py — 48-D touch feature schema.

Vector 48 chiều (thứ tự cố định):
  TAP    (16): tap_n, tap_hold x5, tap_disp x5, tap_iti x5
  SCROLL (23): scroll_n, dur x2, traj x2, sdist x2, vmean x2,
               vmax x2, vlast5 x2, mrl x2, afirst5 x2,
               dir_circ x2, frac x4
  KEY     (9): key_n, key_inter x5, delete_rate, typing_speed, burst_rate
"""
import numpy as np
import pandas as pd
from pathlib import Path

TAP_COLS = [
    "tap_n",
    "tap_hold_mean",   "tap_hold_std",   "tap_hold_median", "tap_hold_p25",   "tap_hold_p75",
    "tap_disp_mean",   "tap_disp_std",   "tap_disp_median", "tap_disp_p25",   "tap_disp_p75",
    "tap_iti_mean",    "tap_iti_std",    "tap_iti_median",  "tap_iti_p25",    "tap_iti_p75",
]
SCROLL_COLS = [
    "scroll_n",
    "scroll_dur_mean",    "scroll_dur_std",
    "scroll_traj_mean",   "scroll_traj_std",
    "scroll_sdist_mean",  "scroll_sdist_std",
    "scroll_vmean_mean",  "scroll_vmean_std",
    "scroll_vmax_mean",   "scroll_vmax_std",
    "scroll_vlast5_mean", "scroll_vlast5_std",
    "scroll_mrl_mean",    "scroll_mrl_std",
    "scroll_afirst5_mean","scroll_afirst5_std",
    "scroll_dir_circmean","scroll_dir_circstd",
    "scroll_frac_up", "scroll_frac_down", "scroll_frac_left", "scroll_frac_right",
]
KEY_COLS = [
    "key_n",
    "key_inter_mean", "key_inter_std", "key_inter_median", "key_inter_p25", "key_inter_p75",
    "key_delete_rate", "key_typing_speed", "key_burst_rate",
]

FEATURE_COLS = TAP_COLS + SCROLL_COLS + KEY_COLS
FEAT_DIM     = len(FEATURE_COLS)
assert FEAT_DIM == 48

_csv_cache: dict[Path, pd.DataFrame | None] = {}

def _load_csv(user_dir: Path):
    if user_dir in _csv_cache:
        return _csv_cache[user_dir]
    p = user_dir / "touch_session_features.csv"
    if not p.exists():
        _csv_cache[user_dir] = None
        return None
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print(f"  Warning: {p}: {e}")
        _csv_cache[user_dir] = None
        return None
    if len(df) == 0 or "session_id" not in df.columns:
        _csv_cache[user_dir] = None
        return None
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    _csv_cache[user_dir] = df
    return df

def clear_cache():
    _csv_cache.clear()

def strip_user_prefix(prefixed: str, user_id: str) -> str:
    pre = f"{user_id}_"
    return prefixed[len(pre):] if prefixed.startswith(pre) else prefixed

def build_session_features(user_dir: Path, session_ids):
    df = _load_csv(user_dir)
    if df is None:
        return None
    targets = {str(s) for s in session_ids}
    matched = df[df["session_id"].astype(str).isin(targets)]
    if len(matched) == 0:
        return None
    mat = matched[FEATURE_COLS].to_numpy(dtype=np.float64)
    vec = mat[0] if len(mat) == 1 else mat.mean(axis=0)
    if np.any(np.isnan(vec)):
        vec = np.nan_to_num(vec, nan=0.0)
    return vec


In [ ]:
%%writefile metrics.py
"""metrics.py — AUC, EER, FAR, FRR."""
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve


def compute_auc(y_true, scores) -> float:
    try:
        return float(roc_auc_score(y_true, scores))
    except ValueError:
        return 0.5


def _clean_roc(y_true, scores):
    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=1)
    mask = np.isfinite(thresholds)
    return fpr[mask], tpr[mask], thresholds[mask]


def compute_eer(y_true, scores) -> float:
    fpr, tpr, thresholds = _clean_roc(y_true, scores)
    if len(fpr) < 2:
        return 0.5
    fnr = 1.0 - tpr
    diffs = fpr - fnr
    for i in range(len(diffs) - 1):
        if diffs[i] * diffs[i + 1] <= 0:
            d0, d1 = diffs[i], diffs[i + 1]
            if d0 == d1:
                eer = (fpr[i] + fnr[i]) / 2
            else:
                t = d0 / (d0 - d1)
                eer = fpr[i] + t * (fpr[i + 1] - fpr[i])
            return float(eer)
    idx = np.argmin(np.abs(diffs))
    return float((fpr[idx] + fnr[idx]) / 2)


def compute_far_frr_at_threshold(y_true, scores, threshold):
    preds = (scores >= threshold).astype(int)
    tp = int(((preds == 1) & (y_true == 1)).sum())
    fp = int(((preds == 1) & (y_true == 0)).sum())
    fn = int(((preds == 0) & (y_true == 1)).sum())
    tn = int(((preds == 0) & (y_true == 0)).sum())
    far = fp / (fp + tn + 1e-10)
    frr = fn / (fn + tp + 1e-10)
    return float(far), float(frr)


def find_eer_threshold(y_true, scores) -> float:
    fpr, tpr, thresholds = _clean_roc(y_true, scores)
    if len(thresholds) < 2:
        return float(np.median(scores)) if len(scores) > 0 else 0.5
    fnr = 1.0 - tpr
    diffs = fpr - fnr
    for i in range(len(diffs) - 1):
        if diffs[i] * diffs[i + 1] <= 0:
            d0, d1 = diffs[i], diffs[i + 1]
            t = 0.5 if d0 == d1 else d0 / (d0 - d1)
            val = thresholds[i] + t * (thresholds[i + 1] - thresholds[i])
            if np.isfinite(val):
                return float(val)
            return float(thresholds[i])
    idx = np.argmin(np.abs(diffs))
    res = float(thresholds[idx])
    return res if np.isfinite(res) else float(np.median(scores))


In [ ]:
%%writefile touch_train.py
"""
touch_train.py — Standalone touch RF training + evaluation.

Quy trình mỗi RUN (giống main.py để metrics so sánh được):
  1. Đọc session list của mỗi user từ y_{walking|inertial}.npy
  2. Session-aware split (train/val/test)
  3. Pre-compute touch vector 48-D 1 lần/run
  4. Fit StandardScaler trên union train vectors
  5. Mỗi owner: sample impostor pool → train RF → score val/test
"""
import argparse, json, pickle, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

from touch_features import (
    build_session_features, strip_user_prefix, FEAT_DIM, clear_cache,
)
from metrics import (
    compute_auc, compute_eer, compute_far_frr_at_threshold, find_eer_threshold,
)


# ── Defaults ────────────────────────────────────────────────────
DEFAULT_N_RUNS           = 10
DEFAULT_POOL_SIZE        = 100
DEFAULT_TEST_SIZE        = 0.20
DEFAULT_VAL_SIZE         = 0.125
DEFAULT_N_ESTIMATORS     = 200
DEFAULT_MAX_FEATURES     = "sqrt"
DEFAULT_CLASS_WEIGHT     = "balanced"
DEFAULT_MIN_SAMPLES_LEAF = 1
DEFAULT_CONTEXT_MODE     = "all"
DEFAULT_SEED_BASE        = 42


def load_session_lists(data_dir: Path, context_mode: str) -> dict:
    y_name = "y_walking.npy" if context_mode == "walking" else "y_inertial.npy"
    out = {}
    for user_dir in sorted(data_dir.iterdir()):
        if not user_dir.is_dir():
            continue
        y_path = user_dir / y_name
        if not y_path.exists():
            print(f"  [skip] {user_dir.name}: thieu {y_name}")
            continue
        sess = np.load(y_path, allow_pickle=True)
        out[user_dir.name] = list(np.unique(sess.astype(str)))
    return out


def split_sessions(sessions_per_user, seed, test_size, val_size):
    rng = np.random.default_rng(seed)
    train_sess, val_sess, test_sess = {}, {}, {}
    for uid, sessions in sessions_per_user.items():
        sessions = np.array(sessions)
        if len(sessions) < 3:
            train_sess[uid] = list(sessions); val_sess[uid] = []; test_sess[uid] = []
            continue
        rng.shuffle(sessions)
        n_te  = max(1, int(len(sessions) * test_size))
        n_val = max(1, int((len(sessions) - n_te) * val_size))
        test_sess[uid]  = list(sessions[:n_te])
        val_sess[uid]   = list(sessions[n_te:n_te + n_val])
        train_sess[uid] = list(sessions[n_te + n_val:])
    return train_sess, val_sess, test_sess


def touch_vectors_for_sessions(data_dir, user_id, session_list):
    user_dir = data_dir / user_id
    vecs = []
    for s in session_list:
        unprefixed = strip_user_prefix(s, user_id)
        vec = build_session_features(user_dir, {unprefixed})
        if vec is not None:
            vecs.append(vec)
    if not vecs:
        return np.zeros((0, FEAT_DIM), dtype=np.float64)
    return np.asarray(vecs, dtype=np.float64)


def precompute(data_dir, sessions_dict):
    return {uid: touch_vectors_for_sessions(data_dir, uid, sess)
            for uid, sess in sessions_dict.items()}


def train_eval_owner(owner_id, train_by_uid, val_by_uid, test_by_uid,
                     scaler, pool_size, n_estimators, max_features,
                     class_weight, min_samples_leaf, seed):
    own_tr = train_by_uid.get(owner_id, np.zeros((0, FEAT_DIM)))
    if len(own_tr) == 0:
        return None, None

    pool_parts = [v for u, v in train_by_uid.items()
                  if u != owner_id and len(v) > 0]
    if not pool_parts:
        return None, None
    pool_all = np.concatenate(pool_parts)
    rng = np.random.default_rng(seed)
    if len(pool_all) > pool_size:
        idx = rng.choice(len(pool_all), size=pool_size, replace=False)
        pool_all = pool_all[idx]

    own_tr_s = scaler.transform(own_tr)
    pool_s   = scaler.transform(pool_all)
    X = np.concatenate([own_tr_s, pool_s])
    y = np.concatenate([np.ones(len(own_tr_s), dtype=np.int32),
                        np.zeros(len(pool_s),  dtype=np.int32)])

    rf = RandomForestClassifier(
        n_estimators=n_estimators, max_features=max_features,
        class_weight=class_weight, min_samples_leaf=min_samples_leaf,
        random_state=seed, n_jobs=-1,
    )
    rf.fit(X, y)

    def _xy(by_uid):
        X_list, y_list = [], []
        for uid, vecs in by_uid.items():
            if len(vecs) == 0:
                continue
            X_list.append(vecs)
            y_list.append(np.full(len(vecs), 1 if uid == owner_id else 0, dtype=np.int32))
        if not X_list:
            return None, None
        return np.concatenate(X_list), np.concatenate(y_list)

    val_X,  val_y  = _xy(val_by_uid)
    test_X, test_y = _xy(test_by_uid)
    if val_X is None or test_X is None:
        return None, rf

    val_X_s  = scaler.transform(val_X)
    test_X_s = scaler.transform(test_X)
    s_val    = rf.predict_proba(val_X_s)[:, 1].astype(np.float32)
    s_test   = rf.predict_proba(test_X_s)[:, 1].astype(np.float32)

    auc_val  = compute_auc(val_y,  s_val)
    auc_test = compute_auc(test_y, s_test)
    eer_test = compute_eer(test_y, s_test)
    thr      = find_eer_threshold(test_y, s_test)
    far, frr = compute_far_frr_at_threshold(test_y, s_test, thr)

    return dict(
        owner_id=owner_id, auc_val=float(auc_val), auc_test=float(auc_test),
        eer_test=float(eer_test), far_at_eer=float(far), frr_at_eer=float(frr),
        thr=float(thr), n_train_pos=int(len(own_tr)),
        n_train_neg=int(len(pool_all)), n_val=int(len(val_y)),
        n_test=int(len(test_y)),
    ), rf


def run_once(run_idx, seed, data_dir, sessions_per_user, args, last_run):
    print(f"\n-- RUN {run_idx + 1}/{args.n_runs}  (seed={seed}) --")
    clear_cache()
    train_sess, val_sess, test_sess = split_sessions(
        sessions_per_user, seed, args.test_size, args.val_size
    )
    train_by_uid = precompute(data_dir, train_sess)
    val_by_uid   = precompute(data_dir, val_sess)
    test_by_uid  = precompute(data_dir, test_sess)

    all_train = [v for v in train_by_uid.values() if len(v) > 0]
    if not all_train:
        return [], {}
    scaler = StandardScaler().fit(np.concatenate(all_train))

    rows = []
    saved_rfs = {} if (last_run and args.save_rfs) else None

    for owner_id in sessions_per_user:
        try:
            row, rf = train_eval_owner(
                owner_id, train_by_uid, val_by_uid, test_by_uid,
                scaler, args.pool_size, args.n_estimators,
                args.max_features, args.class_weight,
                args.min_samples_leaf, seed,
            )
        except Exception as e:
            print(f"  [error] {owner_id}: {e}")
            continue

        if row is None:
            continue
        row["run"]  = run_idx
        row["seed"] = seed
        rows.append(row)
        if saved_rfs is not None and rf is not None:
            saved_rfs[owner_id] = (rf, scaler)
        if not args.quiet:
            print(f"  {owner_id:8s}  AUC_v={row['auc_val']:.4f}  "
                  f"AUC_t={row['auc_test']:.4f}  EER={row['eer_test']:.4f}  "
                  f"FAR={row['far_at_eer']:.4f}  FRR={row['frr_at_eer']:.4f}")
    return rows, (saved_rfs or {})


def write_report(df, args, elapsed, output_dir):
    cfg = vars(args).copy()
    cfg.pop("data_dir", None); cfg.pop("output_dir", None)
    cfg.pop("save_rfs", None); cfg.pop("quiet", None)
    (output_dir / "config.json").write_text(json.dumps(cfg, indent=2, default=str))

    per_run = df.groupby("run").agg({
        "auc_test":"mean", "eer_test":"mean",
        "far_at_eer":"mean", "frr_at_eer":"mean",
    })
    lines = [
        "=== TOUCH-ONLY REPORT ===",
        "",
        f"Users      : {sorted(df['owner_id'].unique())}",
        f"Runs       : {args.n_runs}",
        f"Total time : {elapsed/60:.1f} min",
        "",
        "Hyperparameters:",
        f"  context_mode      : {args.context_mode}",
        f"  pool_size         : {args.pool_size}",
        f"  n_estimators      : {args.n_estimators}",
        f"  max_features      : {args.max_features}",
        f"  class_weight      : {args.class_weight}",
        f"  min_samples_leaf  : {args.min_samples_leaf}",
        "",
        "Overall (mean +/- std qua runs):",
        f"  auc_test    : {per_run['auc_test'].mean():.4f} +/- {per_run['auc_test'].std():.4f}",
        f"  eer_test    : {per_run['eer_test'].mean():.4f} +/- {per_run['eer_test'].std():.4f}",
        f"  far_at_eer  : {per_run['far_at_eer'].mean():.4f} +/- {per_run['far_at_eer'].std():.4f}",
        f"  frr_at_eer  : {per_run['frr_at_eer'].mean():.4f} +/- {per_run['frr_at_eer'].std():.4f}",
        "",
        "Per-user (median qua runs, sorted by AUC):",
    ]
    per_user = df.groupby("owner_id").agg(
        auc_med  = ("auc_test", "median"),
        auc_mean = ("auc_test", "mean"),
        auc_std  = ("auc_test", "std"),
        eer_med  = ("eer_test", "median"),
        eer_std  = ("eer_test", "std"),
        n        = ("auc_test", "count"),
    ).sort_values("auc_med")
    lines.append(per_user.to_string())
    report = "\n".join(lines)
    (output_dir / "final_report.txt").write_text(report)
    print("\n" + report)


def parse_args():
    p = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    p.add_argument("--data-dir",         default="./processed_data")
    p.add_argument("--output-dir",       default="./results_touch")
    p.add_argument("--n-runs",           type=int, default=DEFAULT_N_RUNS)
    p.add_argument("--context-mode",     choices=["walking", "all"], default=DEFAULT_CONTEXT_MODE)
    p.add_argument("--pool-size",        type=int, default=DEFAULT_POOL_SIZE)
    p.add_argument("--n-estimators",     type=int, default=DEFAULT_N_ESTIMATORS)
    p.add_argument("--max-features",     default=DEFAULT_MAX_FEATURES)
    p.add_argument("--class-weight",     default=DEFAULT_CLASS_WEIGHT)
    p.add_argument("--min-samples-leaf", type=int, default=DEFAULT_MIN_SAMPLES_LEAF)
    p.add_argument("--test-size",        type=float, default=DEFAULT_TEST_SIZE)
    p.add_argument("--val-size",         type=float, default=DEFAULT_VAL_SIZE)
    p.add_argument("--seed-base",        type=int, default=DEFAULT_SEED_BASE)
    p.add_argument("--save-rfs",         action="store_true")
    p.add_argument("--quiet",            action="store_true")
    args = p.parse_args()
    if args.max_features not in ("sqrt", "log2", "auto", None):
        try:    args.max_features = int(args.max_features)
        except ValueError:
            try:    args.max_features = float(args.max_features)
            except ValueError: pass
    if args.class_weight in ("none", "None", ""):
        args.class_weight = None
    return args


def main():
    args = parse_args()
    data_dir   = Path(args.data_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    if not data_dir.exists():
        raise FileNotFoundError(f"DATA_DIR khong ton tai: {data_dir}")

    print("=" * 70)
    print("TOUCH-ONLY TRAINING")
    print("=" * 70)
    for k, v in vars(args).items():
        print(f"  {k:18s}: {v}")

    sessions_per_user = load_session_lists(data_dir, args.context_mode)
    if not sessions_per_user:
        raise RuntimeError("Khong load duoc user nao.")
    print(f"\nLoaded {len(sessions_per_user)} users")

    t0 = time.time()
    all_rows = []
    last_rfs = {}
    for run_idx in range(args.n_runs):
        seed = args.seed_base + run_idx
        rows, rfs = run_once(run_idx, seed, data_dir, sessions_per_user, args,
                             last_run=(run_idx == args.n_runs - 1))
        all_rows.extend(rows)
        if rfs:
            last_rfs = rfs

    elapsed = time.time() - t0
    df = pd.DataFrame(all_rows)
    df.to_csv(output_dir / "summary.csv", index=False)
    print(f"\nSaved {output_dir/'summary.csv'} ({len(df)} rows)")

    if args.save_rfs and last_rfs:
        rfs_dir = output_dir / "rfs"
        rfs_dir.mkdir(exist_ok=True)
        for uid, (rf, scaler) in last_rfs.items():
            with (rfs_dir / f"{uid}.pkl").open("wb") as f:
                pickle.dump({"rf": rf, "scaler": scaler}, f)
        print(f"Saved {len(last_rfs)} RFs to {rfs_dir}")

    write_report(df, args, elapsed, output_dir)
    print(f"\nTotal time: {elapsed/60:.1f} min")


if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
    main()


## §5. Cài dependencies

Không cần torch — chỉ sklearn + pandas.


In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib
print("Done.")


## §6. Kiểm tra data structure

Quick sanity check: mỗi user có `touch_session_features.csv` và `y_inertial.npy` đúng format không.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

p = Path(DATA_DIR)
rows = []
for ud in sorted(p.iterdir()):
    if not ud.is_dir():
        continue
    touch_csv = ud / "touch_session_features.csv"
    y_path    = ud / ("y_walking.npy" if CONTEXT_MODE == "walking" else "y_inertial.npy")

    if not touch_csv.exists():
        rows.append({"user": ud.name, "touch_csv": "MISSING", "n_touch_sess": 0,
                     "y_npy": y_path.exists(), "n_inertial_sess": 0})
        continue

    df_t = pd.read_csv(touch_csv)
    n_touch = len(df_t)
    n_iner = 0
    if y_path.exists():
        y = np.load(y_path, allow_pickle=True)
        n_iner = len(np.unique(y))

    rows.append({"user": ud.name, "touch_csv": "OK", "n_touch_sess": n_touch,
                 "y_npy": y_path.exists(), "n_inertial_sess": n_iner})

inv = pd.DataFrame(rows)
print(inv.to_string(index=False))
print(f"\nTotal touch sessions: {inv['n_touch_sess'].sum()}")
print(f"Users missing touch CSV: {(inv['touch_csv'] == 'MISSING').sum()}")


## §7. Run baseline

Chạy với hyperparam mặc định = đúng giá trị trong `config.py` của pipeline chính (`min_samples_leaf=1`).

Kết quả `auc_test` baseline phải gần như khớp cột `auc_touch` trong `summary.csv` của pipeline chính (~0.974 mean, ~0.021 std). Nếu lệch > 0.005, có discrepancy với main pipeline — check lại split logic.


In [ ]:
import subprocess, sys, time

baseline_dir = f"./results_touch_baseline"
t0 = time.time()
result = subprocess.run([
    sys.executable, "touch_train.py",
    "--data-dir",         DATA_DIR,
    "--output-dir",       baseline_dir,
    "--n-runs",           str(N_RUNS),
    "--context-mode",     CONTEXT_MODE,
    "--pool-size",        str(BASE_HP["pool_size"]),
    "--n-estimators",     str(BASE_HP["n_estimators"]),
    "--max-features",     str(BASE_HP["max_features"]),
    "--class-weight",     str(BASE_HP["class_weight"]),
    "--min-samples-leaf", "1",
    "--quiet",
], capture_output=False)
print(f"\nBaseline elapsed: {(time.time()-t0)/60:.1f} min  (exit={result.returncode})")


## §8. Hyperparameter ablation — `min_samples_leaf`

Loop qua `MSL_GRID`, mỗi giá trị output sang folder riêng. Mỗi config mất ~1–3 phút.

Mục đích: tìm `min_samples_leaf` giảm được std của AUC/EER mà không hi sinh mean đáng kể.


In [ ]:
import subprocess, sys, time

for msl in MSL_GRID:
    out_dir = f"./results_touch_msl{msl}"
    print("\n" + "=" * 70)
    print(f"  MIN_SAMPLES_LEAF = {msl}")
    print("=" * 70)
    t0 = time.time()
    result = subprocess.run([
        sys.executable, "touch_train.py",
        "--data-dir",         DATA_DIR,
        "--output-dir",       out_dir,
        "--n-runs",           str(N_RUNS),
        "--context-mode",     CONTEXT_MODE,
        "--pool-size",        str(BASE_HP["pool_size"]),
        "--n-estimators",     str(BASE_HP["n_estimators"]),
        "--max-features",     str(BASE_HP["max_features"]),
        "--class-weight",     str(BASE_HP["class_weight"]),
        "--min-samples-leaf", str(msl),
        "--quiet",
    ], capture_output=False)
    print(f"\nMSL={msl} elapsed: {(time.time()-t0)/60:.1f} min  (exit={result.returncode})")


## §9. So sánh kết quả ablation

Đọc tất cả `summary.csv` đã sinh, build bảng so sánh: mean / std của AUC và EER cho từng MSL.

Khuyến nghị: chọn MSL có **EER thấp + std nhỏ**, nếu cùng EER thì ưu tiên std nhỏ (model ổn định hơn quan trọng hơn 0.001 EER trên paper).


In [ ]:
import pandas as pd
from pathlib import Path

rows = []
for msl in MSL_GRID:
    out_dir = Path(f"./results_touch_msl{msl}")
    summary_path = out_dir / "summary.csv"
    if not summary_path.exists():
        print(f"  ⊘  {summary_path} không tồn tại — skip")
        continue
    df = pd.read_csv(summary_path)

    # Aggregate: mean qua user trong từng run, rồi mean ± std qua runs
    per_run = df.groupby("run").agg({
        "auc_test":   "mean",
        "eer_test":   "mean",
        "far_at_eer": "mean",
        "frr_at_eer": "mean",
    })

    rows.append({
        "min_samples_leaf": msl,
        "auc_mean":  round(per_run["auc_test"].mean(), 4),
        "auc_std":   round(per_run["auc_test"].std(),  4),
        "eer_mean":  round(per_run["eer_test"].mean(), 4),
        "eer_std":   round(per_run["eer_test"].std(),  4),
        "far_mean":  round(per_run["far_at_eer"].mean(), 4),
        "frr_mean":  round(per_run["frr_at_eer"].mean(), 4),
        "n_users_below_eer_0.05": int((df.groupby('owner_id')['eer_test'].median() > 0.05).sum()),
    })

cmp_df = pd.DataFrame(rows).set_index("min_samples_leaf")
print("BẢNG SO SÁNH MSL (mean ± std qua runs):")
print(cmp_df.to_string())

# Đề xuất
if len(cmp_df) > 0:
    best_eer_idx = cmp_df["eer_mean"].idxmin()
    best_std_idx = cmp_df["eer_std"].idxmin()
    print(f"\n  Best EER  : MSL = {best_eer_idx}  (eer_mean = {cmp_df.loc[best_eer_idx, 'eer_mean']})")
    print(f"  Best STD  : MSL = {best_std_idx}  (eer_std  = {cmp_df.loc[best_std_idx, 'eer_std']})")


## §10. Per-user boxplot

Vẽ boxplot AUC và EER cho mỗi MSL trên cùng 1 biểu đồ — dễ thấy MSL nào tail dài (nhiều outlier kém) vs MSL nào ổn định.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gom data: per-user MEDIAN qua runs cho mỗi MSL
data_auc, data_eer, labels = [], [], []
for msl in MSL_GRID:
    summary_path = Path(f"./results_touch_msl{msl}") / "summary.csv"
    if not summary_path.exists():
        continue
    df = pd.read_csv(summary_path)
    per_user = df.groupby("owner_id").agg(auc=("auc_test","median"), eer=("eer_test","median"))
    data_auc.append(per_user["auc"].values)
    data_eer.append(per_user["eer"].values)
    labels.append(f"MSL={msl}")

axes[0].boxplot(data_auc, labels=labels)
axes[0].set_title("AUC test per user (median qua runs)")
axes[0].set_ylabel("AUC")
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(data_eer, labels=labels)
axes[1].set_title("EER test per user (median qua runs)")
axes[1].set_ylabel("EER")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("./touch_ablation_boxplot.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved touch_ablation_boxplot.png")


## §11. Per-user table cho MSL tốt nhất

Chọn MSL muốn inspect chi tiết — xem user nào còn vấn đề.


In [ ]:
import pandas as pd
from pathlib import Path

# CHỌN MSL muốn inspect ↓↓↓
MSL_TO_INSPECT = 1  # hoặc 2, 3, 5

summary_path = Path(f"./results_touch_msl{MSL_TO_INSPECT}") / "summary.csv"
df = pd.read_csv(summary_path)

per_user = df.groupby("owner_id").agg(
    auc_mean  = ("auc_test", "mean"),
    auc_std   = ("auc_test", "std"),
    auc_med   = ("auc_test", "median"),
    eer_mean  = ("eer_test", "mean"),
    eer_std   = ("eer_test", "std"),
    eer_med   = ("eer_test", "median"),
    n_runs    = ("auc_test", "count"),
).sort_values("eer_med", ascending=False).round(4)

print(f"PER-USER METRICS — MSL = {MSL_TO_INSPECT}")
print(f"(sorted by eer_med DESC — khó nhất trên đầu)\n")
print(per_user.to_string())


## §12. Save trained RFs (optional)

Chạy lại với `--save-rfs` để pickle từng owner RF — debug / production deployment.


In [ ]:
# Chỉnh MSL_FINAL theo kết quả §9
MSL_FINAL = 1  # ← thay bằng giá trị thắng cuộc của bạn

import subprocess, sys
subprocess.run([
    sys.executable, "touch_train.py",
    "--data-dir",         DATA_DIR,
    "--output-dir",       f"./results_touch_final_msl{MSL_FINAL}",
    "--n-runs",           str(N_RUNS),
    "--context-mode",     CONTEXT_MODE,
    "--n-estimators",     str(BASE_HP["n_estimators"]),
    "--min-samples-leaf", str(MSL_FINAL),
    "--save-rfs",
])


## §13. Download artifacts

Đóng gói các folder kết quả thành 1 zip để tải về.


In [ ]:
import os, zipfile
from pathlib import Path

zip_path = "/content/touch_training_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # Source code
    for f in ["touch_features.py", "metrics.py", "touch_train.py"]:
        if os.path.exists(f):
            zf.write(f, arcname=f)
    # Results folders
    for d in os.listdir("."):
        if d.startswith("results_touch_") and os.path.isdir(d):
            for root, _, files in os.walk(d):
                for f in files:
                    fp = os.path.join(root, f)
                    zf.write(fp, arcname=fp)
    # Plot
    if os.path.exists("touch_ablation_boxplot.png"):
        zf.write("touch_ablation_boxplot.png")

print(f"Zip size: {os.path.getsize(zip_path) // 1024} KB")
print(f"Path    : {zip_path}")


In [ ]:
# Download về máy
try:
    from google.colab import files
    files.download("/content/touch_training_artifacts.zip")
except ImportError:
    print("Không phải Colab — tự copy /content/touch_training_artifacts.zip")


In [ ]:
# Hoặc copy vào Drive
import shutil
DRIVE_OUT = "/content/drive/MyDrive/touch_training_artifacts.zip"
try:
    shutil.copy("/content/touch_training_artifacts.zip", DRIVE_OUT)
    print(f"Đã copy: {DRIVE_OUT}")
except FileNotFoundError:
    print("Drive chưa mount — chạy lại §2.")


## §14. Bước tiếp theo

Sau khi tìm được `MSL_FINAL` tốt nhất:

**Áp vào pipeline chính**: chỉnh `RF_MIN_SAMPLES_LEAF_TOUCH = <MSL_FINAL>` trong `config.py` của notebook chính (`Active_Auth_Training.ipynb`), rerun để xác nhận fusion AUC/EER cũng cải thiện theo.

**Các knob khác để ablation tiếp** (chỉnh `MSL_GRID` hoặc tạo grid mới ở §8):
- `n_estimators`: thử 100, 500 — nhiều cây có giảm variance không
- `max_features`: thử `log2`, `0.5` — feature subsample mạnh hơn
- `class_weight`: thử `None` thay `balanced` — pool ≈ owner sample size đã khá cân
- `pool_size`: thử 50, 200, 500 — pool to hơn có nâng decision boundary không

**Đưa touch_train.py vào pipeline chính**: nếu muốn refactor sạch, copy module này vào `Active_Auth_Training.ipynb` và thay phần train RF touch inline trong `per_user_eval.evaluate_owner` bằng cách gọi `train_eval_owner()` từ đây.
